In [ ]:
import pandas as pd
from datasets import load_dataset

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "MIG-0fb6bae2-fd70-5e49-921b-9d8c9f43e593" #"MIG-03c7a8b7-dd9c-5bda-82fb-1cf2e9c3a34d"
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

/scratch/s3799042/venvs/think-reduction/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/scratch/s3799042/venvs/think-reduction/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1
NVIDIA A100-SXM4-80GB MIG 3g.40gb


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_path = "l3lab/L1-Qwen3-8B-Max"
model_LCPO = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map=device
)
tokenizer = AutoTokenizer.from_pretrained(model_path)
model_LCPO.eval()


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:22<00:00, 11.45s/it]


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
        (post_attention_la

In [ ]:
def generate_hidden_states_cache(df, model, prompt_column_name="prompt", think_step_by_step=True):

    states = []
    cache = {}

    for _, q in tqdm(df.iterrows(), total=len(df), desc="Processing rows"):

        if think_step_by_step:
            prompt = f"{q[prompt_column_name]} Let’s think step by step inside and output the final answer within boxed{{}}."
        else:
            prompt = q[prompt_column_name]

        if prompt in cache:
            states.append(cache[prompt])
            continue

        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model_LCPO(**inputs, output_hidden_states=True)
            hidden = outputs.hidden_states[-1][:, -1, :]

        hidden = (
            hidden.detach()
            .to(torch.float16)
            .cpu()
            .numpy()
            .reshape(-1)
        )

        cache[prompt] = hidden
        states.append(hidden)

        # free GPU memory
        del inputs, outputs
        torch.cuda.empty_cache()

    states = np.stack(states, axis=0)
    df["hidden"] = list(states)

    return df

In [ ]:
df_hidden_state_sbs = generate_hidden_states(df_math_aime, prompt_column_name = "problem")

In [ ]:
df_hidden_state_sbs['problem'].iloc[0]

In [ ]:
from pathlib import Path
df_hidden_state_sbs.to_parquet(Path().resolve().parent.parent / "processed" / "hidden_states_math_aime_with_sbs.parquet")

In [6]:
from pathlib import Path
import pandas as pd

path_train_targets = Path().resolve().parent.parent / "old" / "dataset_splitting" / "train.parquet"
df_train_targets = pd.read_parquet(path_train_targets)

In [7]:
df_targets_hidden = generate_hidden_states_cache(df_train_targets, prompt_column_name = "prompt", think_step_by_step=False)

Processing rows: 100%|██████████| 253660/253660 [10:15<00:00, 411.83it/s]


In [9]:
save_path = Path().resolve().parent.parent / "processed" /"train"/ "train_targets_hidden.parquet"
save_path

PosixPath('/home/s3799042/projects/llm-think-too-much/data/processed/train/train_targets_hidden.parquet')

In [11]:
df_targets_hidden

,question_id,prompt,solution_col,generated_think_text,generated_text,target_think_tokens,generated_think_tokens,latency_sec,is_correct,level,hidden
0,0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...","For the piecewise function to be continuous, t...","Okay, set limits at x=2 and x=-2. Solve for a ...","Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",100,22,0.928551,False,Level 5,"[0.252, 1.3125, -0.758, 0.668, -1.4375, 1.492,..."
1,0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...","For the piecewise function to be continuous, t...","Okay, for continuity at x=2 and x=-2, set limi...","Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",357,90,0.928551,False,Level 5,"[0.252, 1.3125, -0.758, 0.668, -1.4375, 1.492,..."
2,0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...","For the piecewise function to be continuous, t...","Okay, to ensure continuity, the function's lef...","Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",615,182,0.928551,True,Level 5,"[0.252, 1.3125, -0.758, 0.668, -1.4375, 1.492,..."
3,0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...","For the piecewise function to be continuous, t...","Okay, so I need to find a and b to make the fu...","Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",873,283,0.928551,True,Level 5,"[0.252, 1.3125, -0.758, 0.668, -1.4375, 1.492,..."
4,0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...","For the piecewise function to be continuous, t...","Okay, so I need to find a and b such that the ...","Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",1131,460,0.928551,True,Level 5,"[0.252, 1.3125, -0.758, 0.668, -1.4375, 1.492,..."
...,...,...,...,...,...,...,...,...,...,...,...
253655,aime_932,Find the number of rectangles that can be form...,315,"Okay, so I need to figure out how many rectang...",Find the number of rectangles that can be form...,3968,2609,1.130943,False,aime,"[-1.125, 0.2207, -1.055, 0.2109, -1.992, 2.5, ..."
253656,aime_932,Find the number of rectangles that can be form...,315,"Okay, so I need to figure out how many rectang...",Find the number of rectangles that can be form...,4226,3309,1.130943,False,aime,"[-1.125, 0.2207, -1.055, 0.2109, -1.992, 2.5, ..."
253657,aime_932,Find the number of rectangles that can be form...,315,"Okay, so I need to figure out how many rectang...",Find the number of rectangles that can be form...,4484,2874,1.130943,False,aime,"[-1.125, 0.2207, -1.055, 0.2109, -1.992, 2.5, ..."
253658,aime_932,Find the number of rectangles that can be form...,315,"Okay, so I need to figure out how many rectang...",Find the number of rectangles that can be form...,4742,3612,1.130943,False,aime,"[-1.125, 0.2207, -1.055, 0.2109, -1.992, 2.5, ..."


In [10]:
df_targets_hidden.to_parquet(save_path)

In [12]:
#TEST
path_test_targets = Path().resolve().parent.parent / "old" / "dataset_splitting" / "test.parquet"
df_test_targets = pd.read_parquet(path_test_targets)

In [13]:
df_test_hidden = generate_hidden_states_cache(df_test_targets, prompt_column_name = "prompt", think_step_by_step=False)

Processing rows: 100%|██████████| 15000/15000 [00:39<00:00, 379.96it/s]


In [18]:
save_path_test = Path().resolve().parent.parent / "processed" /"train"/ "test_targets_hidden.parquet"
df_test_hidden.to_parquet(save_path_test)